In [8]:
import mlflow
import optuna
import mlflow.sklearn
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

dagshub_token = os.getenv("DAGSHUB_PAT")

if not dagshub_token:
    raise EnvironmentError("DAGSHUB_PAT environment variable is not set")

# DagsHub credentials for MLflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "rajeshxdatascience"
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

# MLflow tracking URI
repo_owner = "rajeshxdatascience"
repo_name = "yt-comment-sentiment-analysis"

mlflow.set_tracking_uri(
    f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow"
)

In [10]:
df = pd.read_csv(r"C:\Users\rajes\Desktop\yt-comment-sentiment-analysis\data\processed\reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [11]:
df.shape

(36662, 2)

In [12]:
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/bf99c39669094a928ed7fc583f2f5362', creation_time=1786389628230, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1786389628230, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [13]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN

df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3) # Trigram
max_features = 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Step 5: Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLFLOW
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algoritm_comparison")

        # Log algorithm name as a paramter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
                
        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)
                
        # Log Classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
                        
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                     mlflow.log_metric(f"{label}_{metric}", value)
                
        # Log the model
        mlflow.sklearn.log_model(model,f"{model_name}_model",skops_trusted_types=["scipy.sparse._csr.csr_matrix"])

# Step 6: Optuna objective function for LogisticRegression
def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors',3,30)
    weights = trial.suggest_categorical('weights',['uniform', 'distance'])
    metric = trial.suggest_categorical('metric',['euclidean', 'manhattan'])


    model = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, metric=metric)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))

# Step 7: Run Optuna for LogisticRegression, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_knn, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = KNeighborsClassifier(n_neighbors=best_params['n_neighbors'], weights=best_params['weights'], metric=best_params['metric'])

    # Log the best model with MLflow, passing the algo_name as "LogisticRegression"
    log_mlflow("KNN", best_model, X_train, X_test, y_train, y_test)


# Run the experiment for LogisticRegression
run_optuna_experiment()

[I 2026-08-13 21:03:58,915] A new study created in memory with name: no-name-f1705735-5a57-4af8-87e0-dbee324845d7
[I 2026-08-13 21:04:01,176] Trial 0 finished with value: 0.4110124709363771 and parameters: {'n_neighbors': 17, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 0 with value: 0.4110124709363771.
[I 2026-08-13 21:04:03,441] Trial 1 finished with value: 0.41418304798139927 and parameters: {'n_neighbors': 16, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 1 with value: 0.41418304798139927.
[I 2026-08-13 21:04:07,569] Trial 2 finished with value: 0.5845487212005919 and parameters: {'n_neighbors': 14, 'weights': 'distance', 'metric': 'euclidean'}. Best is trial 2 with value: 0.5845487212005919.
[I 2026-08-13 21:04:12,043] Trial 3 finished with value: 0.5655252589304587 and parameters: {'n_neighbors': 23, 'weights': 'distance', 'metric': 'euclidean'}. Best is trial 2 with value: 0.5845487212005919.
[I 2026-08-13 21:04:15,898] Trial 4 finished with value:

🏃 View run KNN_SMOTE_TFIDF_Trigrams at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/5/runs/52c32ceae65546bfba335d0fba9b721d
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/5
